# RAG

In [ ]:
!pip install sentence-transformers faiss-cpu
pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu132

In [10]:
# Import libraries
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from pathlib import Path

In [ ]:
QWEN_CODER_WITHOUT_ADAPTER = Path("/home/nguyen/.cache/huggingface/hub/models--unsloth--Qwen2.5-Coder-1.5B-bnb-4bit/snapshots/8e7c25d88b601ed8c67058751f5f6bd03f7538a8")

model_path = QWEN_CODER_WITHOUT_ADAPTER

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

# Documents
documents = [
    "I'm Qwen 2.5 Coder with 1.5B parameters, Fesnguyen download and using me for helping he learning about LLM",
    "Fesnguyen is the owner of me, using raw model from unsloth",
    "I'm here for helping Fesnguyen learning about RAG",
]

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 2053.22it/s]


In [23]:
# Embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Convert documents into vectors
doc_embeddings = embedder.encode(documents)

# Build vector database
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings, dtype=np.float32))

print('Documents indexed:', index.ntotal)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18299.45it/s]


Documents indexed: 3


In [6]:
def retrieve(query, top_k=2):
    query_embedding = embedder.encode([query])

    distances, indices = index.search(
        np.array(query_embedding, dtype=np.float32),
        top_k
    )

    results = [documents[i] for i in indices[0]]
    return results

In [25]:
query = "What are you doing?"

retrieved_docs = retrieve(query)
print(f'retrived_docs: {retrieved_docs}')

context = "\n".join(retrieved_docs)

prompt = f"""
Answer the question using the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

print(prompt)

retrived_docs: ["I'm here for helping Fesnguyen learning about RAG", "I'm Qwen 2.5 Coder with 1.5B parameters, Fesnguyen download and using me for helping he learning about LLM"]

Answer the question using the provided context.

Context:
I'm here for helping Fesnguyen learning about RAG
I'm Qwen 2.5 Coder with 1.5B parameters, Fesnguyen download and using me for helping he learning about LLM

Question:
What are you doing?

Answer:



In [16]:
# =========================================================
# 4. GENERATION FUNCTION
# =========================================================

def generate(prompt, max_new_tokens=200):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=False, # do_sample=False = always pick the most likely next token; do_sample=True = randomly sample possible next tokens.

            # EOS
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return text

In [26]:
response = generate(prompt)

print(response)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/nguyen/micromamba/envs/llm_env/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


I am a language model, specifically a large language model (LLM) with 1.5 billion parameters. I am designed to assist users in various tasks, including learning about LLMs. In this case, I am assisting Fesnguyen in learning about RAG (Relevant and Augmented Generation) by providing him with information and resources to help him understand the concept better.
